# 13 — Advanced Quantitative Analysis

Notebook này phân tích tương quan nâng cao, momentums và beta của rổ tài sản.

In [13]:
# Cell: Setup Project Root
import os
import sys
from pathlib import Path


def find_project_root(start_path=None):
    path = Path(start_path or os.getcwd()).resolve()
    for parent in [path, *path.parents]:
        if (parent / "src").is_dir() and (parent / "config").is_dir():
            return parent
    raise FileNotFoundError("Khong tim thay project root co thu muc src/ va config/")


project_root = find_project_root()
os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project root:", project_root)


Project root: /Users/jamesb/Desktop/vietnam-quant-research


In [14]:
# Cell: Import Advanced Quant Modules
import pandas as pd

from src.analytics.advanced import (
    add_cross_sectional_momentum_rank,
    compute_rolling_beta,
    correlations_by_regime,
    pair_divergence_zscore,
    screen_pair_divergences,
)
from src.analytics.correlation import build_return_matrix, compute_correlation_matrix, rolling_pair_correlation, top_correlation_pairs
from src.data.loaders import read_parquet
from src.reporting.chart_factory import (
    beta_chart,
    correlation_heatmap,
    momentum_ranking_chart,
    pair_divergence_chart,
    rolling_correlation_chart,
)


In [15]:
# Cell: Load Analysis Frame
# CUSTOMIZABLE: Define your custom watchlist here to run advanced quant and correlation diagnostics, or leave as None
user_watchlist = ["SSI", "HPG", "FPT", "VIC", "VNM", "TCB", "MBB", "MWG", "DGC", "PNJ"]  # e.g., ["SSI", "HPG", "FPT", "VIC", "VNM", "TCB", "MBB", "MWG", "DGC", "PNJ"]
analysis_df = read_parquet("data/signals/setup_signals.parquet")
volatility_df = read_parquet("data/processed/stock_features_volatility.parquet")
distribution_df = read_parquet("data/processed/stock_features_distribution.parquet")
if analysis_df.empty:
    analysis_df = volatility_df.copy()
if analysis_df.empty:
    analysis_df = distribution_df.copy()
if analysis_df.empty:
    raise FileNotFoundError(
        "Khong tim thay input cho notebook 10. Hay tao truoc setup_signals.parquet hoac stock_features_volatility.parquet"
    )
if "date" not in analysis_df.columns:
    raise KeyError(f"analysis_df khong co cot date. Current columns: {analysis_df.columns.tolist()}")
if "symbol" not in analysis_df.columns:
    raise KeyError(f"analysis_df khong co cot symbol. Current columns: {analysis_df.columns.tolist()}")
analysis_df["date"] = pd.to_datetime(analysis_df["date"])
for enrich_df, enrich_cols in [
    (volatility_df, ["volatility_regime", "volatility_percentile", "atr_pct_14"]),
    (distribution_df, ["dist_skew_60d", "dist_kurt_60d", "tail_ratio_60d"]),
]:
    if enrich_df.empty:
        continue
    enrich_df = enrich_df.copy()
    enrich_df["date"] = pd.to_datetime(enrich_df["date"])
    missing_cols = [col for col in enrich_cols if col in enrich_df.columns and col not in analysis_df.columns]
    if missing_cols:
        analysis_df = analysis_df.merge(
            enrich_df[["symbol", "date", *missing_cols]],
            on=["symbol", "date"],
            how="left",
        )
if user_watchlist is not None:
    watchlist_upper = [sym.upper() for sym in user_watchlist]
    analysis_df = analysis_df[analysis_df["symbol"].str.upper().isin(watchlist_upper)].copy()
    print(f"Using user custom watchlist: {watchlist_upper}")
analysis_df = analysis_df.sort_values(["symbol", "date"]).copy()
print("Loaded analysis_df columns:", analysis_df.columns.tolist())
analysis_df.shape

Using user custom watchlist: ['SSI', 'HPG', 'FPT', 'VIC', 'VNM', 'TCB', 'MBB', 'MWG', 'DGC', 'PNJ']
Loaded analysis_df columns: ['symbol', 'market', 'open', 'high', 'low', 'close', 'volume', 'value', 'date', 'suspect_corporate_action', 'return_1d', 'return_5d', 'forward_return_5d', 'return_10d', 'forward_return_10d', 'return_20d', 'forward_return_20d', 'return_60d', 'forward_return_60d', 'log_return_1d', 'ma20', 'distance_ma20', 'above_ma20', 'ma50', 'distance_ma50', 'above_ma50', 'ma200', 'distance_ma200', 'above_ma200', 'true_range', 'atr_14', 'atr_pct_14', 'realized_vol_10d', 'realized_vol_20d', 'realized_vol_60d', 'ewma_vol_20d', 'volatility_percentile', 'volatility_regime', 'dist_mean_60d', 'dist_median_60d', 'dist_skew_60d', 'dist_kurt_60d', 'dist_q05_60d', 'dist_q95_60d', 'tail_ratio_60d', 'volume_ma20', 'volume_zscore_20d', 'value_ma20', 'value_zscore_20d', 'breakout_20d', 'pullback_ma20', 'volume_spike']


(16334, 52)

In [16]:
# Cell: Build Correlation Views
return_matrix = build_return_matrix(analysis_df, return_col="return_1d")
corr_matrix = compute_correlation_matrix(return_matrix, min_periods=40)
top_pairs = top_correlation_pairs(corr_matrix, top_n=15, min_abs_correlation=0.4)
display(top_pairs)
correlation_heatmap(corr_matrix, title="Watchlist Return Correlation Heatmap")


,symbol_a,symbol_b,correlation,abs_correlation
0,MBB,TCB,0.776312,0.776312
1,MBB,SSI,0.647488,0.647488
2,HPG,SSI,0.620091,0.620091
3,SSI,TCB,0.602393,0.602393
4,HPG,MBB,0.592056,0.592056
5,MBB,MWG,0.570690,0.570690
6,MWG,TCB,0.554162,0.554162
7,HPG,TCB,0.546855,0.546855
8,FPT,MWG,0.540975,0.540975
9,MWG,SSI,0.532870,0.532870


In [17]:
# Cell: Rolling Correlation And Divergence
pair = top_pairs.iloc[0] if not top_pairs.empty else pd.Series({"symbol_a": "SSI", "symbol_b": "HPG"})

rolling_corr = rolling_pair_correlation(
    analysis_df,
    symbol_a=pair["symbol_a"],
    symbol_b=pair["symbol_b"],
    window=60,
)
divergence = pair_divergence_zscore(
    analysis_df,
    symbol_a=pair["symbol_a"],
    symbol_b=pair["symbol_b"],
    window=60,
)

display(rolling_corr.tail())
display(divergence.tail())
display(rolling_correlation_chart(rolling_corr, title="60-day Rolling Correlation"))
pair_divergence_chart(divergence, title=f"Pair Divergence: {pair['symbol_a']} vs {pair['symbol_b']}")


,date,symbol_a,symbol_b,rolling_correlation
1628,2026-07-16,MBB,TCB,0.402411
1629,2026-07-17,MBB,TCB,0.416205
1630,2026-07-20,MBB,TCB,0.517583
1631,2026-07-21,MBB,TCB,0.513546
1632,2026-07-22,MBB,TCB,0.562548


,date,symbol_a,symbol_b,price_a,price_b,spread,rolling_mean,rolling_std,divergence_zscore
1629,2026-07-16,MBB,TCB,24150.0,31900.0,-0.278322,-0.294193,0.023963,0.662317
1630,2026-07-17,MBB,TCB,23750.0,31450.0,-0.280816,-0.294589,0.023533,0.585236
1631,2026-07-20,MBB,TCB,22800.0,29950.0,-0.272769,-0.294634,0.023487,0.930982
1632,2026-07-21,MBB,TCB,22700.0,29800.0,-0.272143,-0.294646,0.023475,0.958582
1633,2026-07-22,MBB,TCB,22350.0,29000.0,-0.260470,-0.293867,0.023824,1.401857


In [18]:
# Cell: Screen Pair Divergences
divergence_screen = screen_pair_divergences(
    analysis_df,
    corr_matrix=corr_matrix,
    top_n_pairs=10,
    min_abs_correlation=0.4,
    zscore_window=60,
)
divergence_screen


,date,symbol_a,symbol_b,price_a,price_b,spread,rolling_mean,rolling_std,divergence_zscore,correlation,abs_correlation
0,2026-07-22,MBB,SSI,22350.0,22650.0,-0.013334,-0.103482,0.035335,2.551247,0.647488,0.647488
1,2026-07-22,HPG,SSI,20700.0,22650.0,-0.090026,-0.130783,0.016568,2.460010,0.620091,0.620091
2,2026-07-22,HPG,MBB,20700.0,22350.0,-0.076693,-0.027301,0.030154,-1.637985,0.592056,0.592056
3,2026-07-22,MWG,SSI,70600.0,22650.0,1.136870,1.079168,0.035650,1.618579,0.532870,0.532870
4,2026-07-22,SSI,TCB,22650.0,29000.0,-0.247136,-0.190385,0.036689,-1.546813,0.602393,0.602393
5,2026-07-22,MBB,TCB,22350.0,29000.0,-0.260470,-0.293867,0.023824,1.401857,0.776312,0.776312
6,2026-07-22,MBB,MWG,22350.0,70600.0,-1.150204,-1.182650,0.024340,1.333049,0.570690,0.570690
7,2026-07-22,HPG,TCB,20700.0,29000.0,-0.337162,-0.321168,0.035390,-0.451943,0.546855,0.546855
8,2026-07-22,FPT,MWG,64600.0,70600.0,-0.088816,-0.095035,0.043630,0.142556,0.540975,0.540975
9,2026-07-22,MWG,TCB,70600.0,29000.0,0.889734,0.888783,0.030312,0.031388,0.554162,0.554162


In [19]:
# Cell: Cross Sectional Momentum Ranking
momentum_df = add_cross_sectional_momentum_rank(analysis_df, lookback=20)
latest_momentum = momentum_df[momentum_df["date"] == momentum_df["date"].max()].copy()
display(latest_momentum.sort_values("momentum_rank_20d").head(10))
momentum_ranking_chart(latest_momentum, momentum_col="momentum_20d", top_n=10, title="Top 10 Momentum Names")


,symbol,market,open,high,low,close,volume,value,date,suspect_corporate_action,...,volume_zscore_20d,value_ma20,value_zscore_20d,breakout_20d,pullback_ma20,volume_spike,momentum_20d,momentum_pct_rank_20d,momentum_rank_20d,momentum_decile_20d
687472,VNM,HOSE,58700,60100,58400,59100,10506300,6.218116e+11,2026-07-22,False,...,2.004673,2.337214e+11,2.086715,False,False,True,0.046907,1.0,1.0,10.0
373462,MBB,HOSE,22800,23200,22350,22350,21154000,4.788871e+11,2026-07-22,False,...,1.717953,3.382145e+11,1.339605,False,False,False,-0.062697,0.9,2.0,9.0
239071,FPT,HOSE,65000,67500,64600,64600,10060700,6.615788e+11,2026-07-22,False,...,0.523620,5.552687e+11,0.386574,False,False,False,-0.087571,0.8,3.0,8.0
392989,MWG,HOSE,75900,76000,70600,70600,13474000,9.718678e+11,2026-07-22,False,...,3.508940,3.223502e+11,3.394130,False,False,True,-0.092545,0.7,4.0,7.0
565735,TCB,HOSE,29900,30300,29000,29000,19146900,5.683841e+11,2026-07-22,False,...,1.531030,4.234567e+11,1.134651,False,False,False,-0.107692,0.6,5.0,6.0
292954,HPG,HOSE,20900,21050,20500,20700,43471900,9.006188e+11,2026-07-22,False,...,1.946597,4.726835e+11,1.819846,False,False,False,-0.119149,0.5,6.0,5.0
669860,VIC,HOSE,216000,217000,202100,202100,5736200,1.188169e+12,2026-07-22,False,...,2.808348,6.891449e+11,2.607786,False,False,True,-0.123210,0.4,7.0,4.0
546548,SSI,HOSE,23250,23650,22400,22650,34754400,7.927094e+11,2026-07-22,False,...,1.685054,4.840171e+11,1.437374,False,False,False,-0.151685,0.3,8.0,3.0
176487,DGC,HOSE,42900,42900,40500,40500,1267800,5.193778e+10,2026-07-22,False,...,2.468836,2.628408e+10,2.040632,False,False,True,-0.196429,0.2,9.0,2.0
464902,PNJ,HOSE,35500,36950,35500,35500,7538000,2.678452e+11,2026-07-22,False,...,0.292382,2.593831e+11,0.029629,False,False,False,-0.458015,0.1,10.0,1.0


In [20]:
# Cell: Rolling Beta Vs VNINDEX
vnindex_path = Path("data/raw/index_ohlcv/VNINDEX.parquet")
if vnindex_path.exists():
    vnindex = read_parquet(vnindex_path)
    vnindex["date"] = pd.to_datetime(vnindex["date"])
    beta_df = compute_rolling_beta(
        analysis_df,
        benchmark_df=vnindex,
        benchmark_symbol="VNINDEX",
        window=60,
    )
    display(beta_df[["symbol", "date", "rolling_beta_60d"]].dropna().tail(10))
    beta_chart(beta_df, symbol=beta_df["symbol"].iloc[0], beta_col="rolling_beta_60d", title="Rolling Beta vs VNINDEX")
else:
    pd.DataFrame({"info": ["Chua co data/raw/index_ohlcv/VNINDEX.parquet. Them benchmark data de bat rolling beta vs VNINDEX."]})


,symbol,date,rolling_beta_60d
16324,VNM,2026-07-09,0.134152
16325,VNM,2026-07-10,0.128485
16326,VNM,2026-07-13,0.238093
16327,VNM,2026-07-14,0.260440
16328,VNM,2026-07-15,0.236275
16329,VNM,2026-07-16,0.225916
16330,VNM,2026-07-17,0.116736
16331,VNM,2026-07-20,0.139221
16332,VNM,2026-07-21,0.135493
16333,VNM,2026-07-22,0.023432


In [21]:
# Cell: Correlation By Regime
regime_corr = correlations_by_regime(
    analysis_df,
    regimes=["Low Vol", "Normal Vol", "Panic Vol"],
    regime_col="volatility_regime",
    min_periods=20,
)

selected_regime = "Low Vol"
display(regime_corr[selected_regime])
correlation_heatmap(regime_corr[selected_regime], title=f"Correlation Heatmap - {selected_regime}")


symbol,DGC,FPT,HPG,MBB,MWG,PNJ,SSI,TCB,VIC,VNM
symbol,,,,,,,,,,
DGC,1.000000,0.191692,0.097791,0.393448,0.511306,-0.084142,0.047791,0.113950,0.357097,0.299621
FPT,0.191692,1.000000,0.262478,0.210045,0.315205,0.044302,0.318181,0.344793,0.198065,0.148851
HPG,0.097791,0.262478,1.000000,0.365042,0.327563,0.153509,0.424675,0.345138,0.035038,0.273770
MBB,0.393448,0.210045,0.365042,1.000000,0.412228,0.055361,0.458613,0.688916,0.084846,0.164878
MWG,0.511306,0.315205,0.327563,0.412228,1.000000,0.259762,0.367424,0.250394,0.269997,0.298267
PNJ,-0.084142,0.044302,0.153509,0.055361,0.259762,1.000000,0.099323,0.286594,0.245811,0.289706
SSI,0.047791,0.318181,0.424675,0.458613,0.367424,0.099323,1.000000,0.357377,0.131225,0.264075
TCB,0.113950,0.344793,0.345138,0.688916,0.250394,0.286594,0.357377,1.000000,0.095123,0.214291
VIC,0.357097,0.198065,0.035038,0.084846,0.269997,0.245811,0.131225,0.095123,1.000000,0.178073
